# 09 MLP Genetic Algorithm

GA uses the exact same screened domain and 40 unique evaluation budget as Random Search. Duplicate configurations reuse cached fitness.


## 1. Package Setup


In [1]:
# Purpose: Package installs are documented but not run automatically during this static refactor.
# %pip install tensorflow scikit-learn pandas numpy matplotlib seaborn joblib soundfile librosa


## 2. Load Cache and Search Space


In [2]:
import os
from pathlib import Path

# Purpose: Mounts Google Drive when this notebook is running in Google Colab.
# Why this exists: the project files, cached MFCC features, manifests, models, figures,
# and metric outputs live in Google Drive during Colab runs. The /content/drive path
# only represents the real MyDrive files after drive.mount("/content/drive") succeeds.
# Important: the project root should be the folder that contains Data, Model Variants,
# and outputs. For this project, that expected Colab folder is the INM701 folder below.
COLAB_DRIVE_MOUNT_POINT = Path("/content/drive")
EXPECTED_COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")

try:
    from google.colab import drive

    drive.mount(str(COLAB_DRIVE_MOUNT_POINT))
    print("Google Colab detected. Google Drive mounted.")

    current_project_root = os.environ.get("INTRO_AI_PROJECT_ROOT")
    current_data_dir_exists = bool(current_project_root) and (Path(current_project_root) / "Data").exists()
    expected_data_dir_exists = (EXPECTED_COLAB_PROJECT_ROOT / "Data").exists()

    # Purpose: Keep a valid user-provided project root, but repair stale runtime state
    # if a previous cell pointed INTRO_AI_PROJECT_ROOT somewhere that does not contain Data.
    if current_data_dir_exists:
        print("INTRO_AI_PROJECT_ROOT already points to a folder with Data, so it was kept.")
    elif expected_data_dir_exists:
        os.environ["INTRO_AI_PROJECT_ROOT"] = str(EXPECTED_COLAB_PROJECT_ROOT)
        print("INTRO_AI_PROJECT_ROOT set to the expected INM701 project folder.")
    elif not current_project_root:
        os.environ["INTRO_AI_PROJECT_ROOT"] = str(EXPECTED_COLAB_PROJECT_ROOT)
        print("INTRO_AI_PROJECT_ROOT was not set, so it now points to the expected INM701 folder.")
    else:
        print("INTRO_AI_PROJECT_ROOT was kept, but Data was not found there or in the expected INM701 folder.")
except Exception as exc:
    print("Google Colab Drive mount skipped. This is expected outside Colab.")
    print("Mount skip reason:", exc)

active_project_root = os.environ.get("INTRO_AI_PROJECT_ROOT", "not set")
print("INTRO_AI_PROJECT_ROOT:", active_project_root)
if active_project_root != "not set":
    active_project_root = Path(active_project_root)
    print("Project root exists:", active_project_root.exists())
    print("Expected Data folder:", active_project_root / "Data")
    print("Data folder exists:", (active_project_root / "Data").exists())


Mounted at /content/drive
Google Colab detected. Google Drive mounted.
INTRO_AI_PROJECT_ROOT set to the expected INM701 project folder.
INTRO_AI_PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Education/INM701
Project root exists: True
Expected Data folder: /content/drive/MyDrive/Colab Notebooks/Education/INM701/Data
Data folder exists: True


In [3]:
# Purpose: Later notebooks load the one MLP-ready cache made by notebook 00. They do not
# rescan folders, regenerate splits, extract MFCCs, fit scalers, or recalculate class weights.
import hashlib
import json
import os
import random
import time
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 7
EARLY_STOP_MIN_DELTA = 1e-4
THRESHOLD = 0.5
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}
CONFIRMATION_SEEDS = [42, 123, 2026]


def resolve_project_root():
    # Purpose: default path.
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    # Purpose: Runs each candidate configuration under the same data split for a fair validation comparison.
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


# Purpose: Centralizes filesystem paths so dataset inputs, caches, figures, models, and metric tables are easy
# Purpose: to trace.
PROJECT_ROOT = resolve_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mlp"
CACHE_DIR = OUTPUT_DIR / "cache"
MANIFESTS_DIR = OUTPUT_DIR / "manifests"
CONFIGS_DIR = OUTPUT_DIR / "configs"
TABLES_DIR = OUTPUT_DIR / "tables"
HISTORIES_DIR = OUTPUT_DIR / "histories"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
PILOT_LEGACY_DIR = OUTPUT_DIR / "pilot_legacy"
# Purpose: Creates each output directory before later cells try to save tables, figures, or models.
for directory in [OUTPUT_DIR, CACHE_DIR, MANIFESTS_DIR, CONFIGS_DIR, TABLES_DIR, HISTORIES_DIR, METRICS_DIR, FIGURES_DIR, MODELS_DIR, PILOT_LEGACY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    # Purpose: Stops the notebook early with a clear message when a required upstream artifact is missing.
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")
    return path


def load_json(path):
    # Purpose: Keeps the load_json helper isolated so later notebook cells can call it consistently.
    with open(require_file(path), "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(payload, path):
    # Purpose: Keeps the save_json helper isolated so later notebook cells can call it consistently.
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def load_class_weights():
    # Purpose: Keeps the load_class_weights helper isolated so later notebook cells can call it consistently.
    payload = load_json(CONFIGS_DIR / "class_weights.json")
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


for required in [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    CONFIGS_DIR / "class_weights.json",
]:
    require_file(required)

X_train = np.load(CACHE_DIR / "X_train.npy")
y_train = np.load(CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation = np.load(CACHE_DIR / "X_validation.npy")
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(CACHE_DIR / "validation_metadata.csv")
feature_config = load_json(CACHE_DIR / "feature_config.json")
CLASS_WEIGHTS = load_class_weights()

if len(X_train) != len(y_train) or len(X_train) != len(train_metadata):
    raise RuntimeError("Training cache arrays, labels and metadata are not row-aligned.")
if len(X_validation) != len(y_validation) or len(X_validation) != len(validation_metadata):
    raise RuntimeError("Validation cache arrays, labels and metadata are not row-aligned.")

print("Loaded MLP cache:", CACHE_DIR)
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)

search_payload = load_json(CONFIGS_DIR / "screened_hpo_search_space.json")
SEARCH_SPACE = search_payload["search_space"]


Loaded MLP cache: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/mlp/cache
X_train shape: (9060, 80)
X_validation shape: (1942, 80)


## 3. Keras Utilities


In [4]:
# Purpose: Shared beginner-readable Keras utilities for validation-only model selection.
def normalise_config(config):
    # Purpose: Keeps the normalise_config helper isolated so later notebook cells can call it consistently.
    normalised = dict(config)
    normalised["hidden_units"] = [int(value) for value in normalised["hidden_units"]]
    normalised["activation"] = str(normalised["activation"])
    normalised["optimizer"] = str(normalised["optimizer"])
    normalised["dropout"] = float(normalised["dropout"])
    normalised["learning_rate"] = float(normalised["learning_rate"])
    normalised["batch_size"] = int(normalised["batch_size"])
    normalised["loss"] = normalised.get("loss", "binary_crossentropy")
    normalised["threshold"] = float(normalised.get("threshold", THRESHOLD))
    return normalised


def config_json(config):
    # Purpose: Keeps the config_json helper isolated so later notebook cells can call it consistently.
    return json.dumps(normalise_config(config), sort_keys=True)


def config_key(config):
    # Purpose: Keeps the config_key helper isolated so later notebook cells can call it consistently.
    return hashlib.sha256(config_json(config).encode("utf-8")).hexdigest()


def seed_from_config(config, base_seed=RANDOM_STATE):
    # Purpose: Keeps the seed_from_config helper isolated so later notebook cells can call it consistently.
    digest = hashlib.sha256(f"{base_seed}:{config_json(config)}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % (2**31 - 1)


def set_global_seed(seed):
    # Purpose: Keeps the set_global_seed helper isolated so later notebook cells can call it consistently.
    random.seed(int(seed))
    np.random.seed(int(seed))
    tf.keras.utils.set_random_seed(int(seed))
    os.environ["PYTHONHASHSEED"] = str(int(seed))


def make_optimizer(config):
    # Purpose: Keeps the make_optimizer helper isolated so later notebook cells can call it consistently.
    name = str(config["optimizer"]).lower()
    lr = float(config["learning_rate"])
    if name == "adam":
        return tf.keras.optimizers.Adam(learning_rate=lr)
    if name == "rmsprop":
        return tf.keras.optimizers.RMSprop(learning_rate=lr)
    if name in ["sgd", "sgd_momentum"]:
        return tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    raise ValueError(f"Unsupported optimizer: {config['optimizer']}")


def build_mlp_model(config, input_dim):
    # Purpose: Constructs the MLP architecture for the current experiment configuration.
    config = normalise_config(config)
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    # Purpose: Iterates over this collection to build the next table, feature set, or experiment result
    # Purpose: consistently.
    for units in config["hidden_units"]:
        model.add(Dense(units, activation=config["activation"]))
        if config["dropout"] > 0:
            model.add(Dropout(config["dropout"]))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(loss=config["loss"], optimizer=make_optimizer(config), metrics=["accuracy"])
    return model


def train_and_evaluate_config(config, run_seed, run_name, verbose=0):
    # Purpose: consistently.
    config = normalise_config(config)
    tf.keras.backend.clear_session()
    set_global_seed(run_seed)
    model = build_mlp_model(config, X_train.shape[1])
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            min_delta=EARLY_STOP_MIN_DELTA,
            restore_best_weights=True,
        )
    ]
    start = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=MAX_EPOCHS,
        batch_size=config["batch_size"],
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks,
        verbose=verbose,
    )
    runtime = time.perf_counter() - start
    probability = model.predict(X_validation, batch_size=config["batch_size"], verbose=0).ravel()
    pred = (probability >= config["threshold"]).astype(int)
    row = {
        "configuration": config,
        "config_json": config_json(config),
        "config_key": config_key(config),
        "seed": int(run_seed),
        "validation_macro_f1": float(f1_score(y_validation, pred, average="macro", zero_division=0)),
        "validation_binary_f1_synthetic": float(f1_score(y_validation, pred, pos_label=1, zero_division=0)),
        "validation_accuracy": float(accuracy_score(y_validation, pred)),
        "validation_precision_synthetic": float(precision_score(y_validation, pred, pos_label=1, zero_division=0)),
        "validation_recall_synthetic": float(recall_score(y_validation, pred, pos_label=1, zero_division=0)),
        "best_validation_loss": float(np.min(history.history["val_loss"])),
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "epochs_trained": int(len(history.history["loss"])),
        "runtime_seconds": float(runtime),
        "run_name": run_name,
    }
    history_df = pd.DataFrame(history.history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
    return row, history_df


def flat_result(row, extra=None):
    # Purpose: Keeps the flat_result helper isolated so later notebook cells can call it consistently.
    extra = extra or {}
    config = normalise_config(row["configuration"])
    flat = dict(extra)
    flat.update({
        "hidden_units": json.dumps(config["hidden_units"]),
        "activation": config["activation"],
        "optimizer": config["optimizer"],
        "dropout": config["dropout"],
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "loss": config["loss"],
        "threshold": config["threshold"],
        "config_json": row["config_json"],
        "config_key": row["config_key"],
        "seed": row["seed"],
        "validation_macro_f1": row["validation_macro_f1"],
        "validation_binary_f1_synthetic": row["validation_binary_f1_synthetic"],
        "validation_accuracy": row["validation_accuracy"],
        "validation_precision_synthetic": row["validation_precision_synthetic"],
        "validation_recall_synthetic": row["validation_recall_synthetic"],
        "best_validation_loss": row["best_validation_loss"],
        "best_epoch": row["best_epoch"],
        "epochs_trained": row["epochs_trained"],
        "runtime_seconds": row["runtime_seconds"],
    })
    return flat


def select_best(rows):
    # Purpose: Keeps the select_best helper isolated so later notebook cells can call it consistently.
    return max(rows, key=lambda row: (row["validation_macro_f1"], -row["best_validation_loss"]))


## 4. Run or Load GA


In [5]:
# Purpose: Runs or reloads GA HPO; rebuilds the best-config JSON from trial CSVs when needed.
RUN_GENETIC_ALGORITHM = True
MAX_UNIQUE_EVALUATIONS = 40
POPULATION_SIZE = 10
TOURNAMENT_SIZE = 3
ELITE_COUNT = 2
MUTATION_RATE = 0.20
trials_path = TABLES_DIR / "09_genetic_algorithm_trials.csv"
best_path = CONFIGS_DIR / "09_genetic_algorithm_best.json"

def rebuild_best_json_from_trials(trials_path, best_path, method_name):
    # Purpose: Rebuilds a missing best-config JSON from an existing search trial CSV.
    # This keeps the HPO handoff usable without rerunning expensive model training.
    trials_path = Path(trials_path)
    best_path = Path(best_path)
    if not trials_path.exists():
        print("Cannot rebuild HPO best config because trial CSV is missing:", trials_path)
        return None

    trials_df = pd.read_csv(trials_path)
    required_columns = {"config_json", "validation_macro_f1", "best_validation_loss"}
    missing_columns = required_columns - set(trials_df.columns)
    if missing_columns:
        raise RuntimeError(f"Cannot rebuild {best_path.name}; missing columns in {trials_path.name}: {sorted(missing_columns)}")
    if trials_df.empty:
        raise RuntimeError(f"Cannot rebuild {best_path.name}; trial CSV is empty: {trials_path}")

    unique_df = trials_df.drop_duplicates("config_key") if "config_key" in trials_df.columns else trials_df
    best_row = unique_df.sort_values(["validation_macro_f1", "best_validation_loss"], ascending=[False, True]).iloc[0]
    payload = best_row.to_dict()
    payload["method"] = method_name
    payload["configuration"] = normalise_config(json.loads(best_row["config_json"]))
    payload["selection_metric"] = "validation_macro_f1"
    payload["rebuilt_from_trials_csv"] = str(trials_path)
    save_json(payload, best_path)
    print("Rebuilt HPO best config from existing trials:", best_path)
    return payload

SEARCH_KEYS = list(SEARCH_SPACE)
def random_config(rng):
    # Purpose: Samples one valid configuration from the screened HPO search space.
    return normalise_config({key: SEARCH_SPACE[key][int(rng.integers(len(SEARCH_SPACE[key])))] for key in SEARCH_KEYS})
def mutate(config, rng):
    # Purpose: Randomly changes some tunable fields while preserving fixed fields such as
    # loss and threshold.
    child = normalise_config(config)
    for key in SEARCH_KEYS:
        if key not in ["loss", "threshold"] and rng.random() < MUTATION_RATE:
            child[key] = SEARCH_SPACE[key][int(rng.integers(len(SEARCH_SPACE[key])))]
    return normalise_config(child)
def crossover(a, b, rng):
    # Purpose: Combines two parent configurations by choosing each field from one parent.
    return normalise_config({key: a[key] if rng.random() < 0.5 else b[key] for key in SEARCH_KEYS})
def tournament(rows, rng):
    # Purpose: Selects a strong parent from a random subset using the same validation ranking
    # rule as the rest of the MLP workflow.
    indices = rng.choice(len(rows), size=min(TOURNAMENT_SIZE, len(rows)), replace=False)
    return select_best([rows[int(index)] for index in indices])["configuration"]

if RUN_GENETIC_ALGORITHM:
    rng = np.random.default_rng(RANDOM_STATE)
    population = [random_config(rng) for _ in range(POPULATION_SIZE)]
    cache = {}
    trial_rows = []
    generation = 0
    while len(cache) < MAX_UNIQUE_EVALUATIONS:
        generation += 1
        # Purpose: Evaluates each generation and caches repeated configurations so duplicate
        # models do not consume extra training time.
        for individual, config in enumerate(population, start=1):
            key = config_key(config)
            if key in cache:
                row = dict(cache[key])
                row.update({"generation": generation, "individual": individual, "cache_hit": True})
                trial_rows.append(row)
                continue
            number = len(cache) + 1
            row, _ = train_and_evaluate_config(config, seed_from_config(config), f"09_ga_{number:03d}", verbose=2)
            row.update({"method": "genetic_algorithm", "generation": generation, "individual": individual, "unique_evaluation_number": number, "cache_hit": False})
            cache[key] = row
            trial_rows.append(row)
            pd.DataFrame([flat_result(item, {"method": "genetic_algorithm", "generation": item.get("generation"), "individual": item.get("individual"), "unique_evaluation_number": item.get("unique_evaluation_number"), "cache_hit": item.get("cache_hit")}) for item in trial_rows]).to_csv(trials_path, index=False)
            if len(cache) >= MAX_UNIQUE_EVALUATIONS:
                break
        rows = list(cache.values())
        elites = sorted(rows, key=lambda row: (row["validation_macro_f1"], -row["best_validation_loss"]), reverse=True)[:ELITE_COUNT]
        population = [normalise_config(row["configuration"]) for row in elites]
        while len(population) < POPULATION_SIZE:
            population.append(mutate(crossover(tournament(rows, rng), tournament(rows, rng), rng), rng))
    save_json(flat_result(select_best(list(cache.values())), {"method": "genetic_algorithm"}), best_path)
elif best_path.exists():
    best_payload = load_json(best_path)
    print("Loaded HPO best config:", best_path)
    if trials_path.exists():
        display(pd.read_csv(trials_path).sort_values(["validation_macro_f1", "best_validation_loss"], ascending=[False, True]).head())
    else:
        display(pd.DataFrame([best_payload]))
elif trials_path.exists():
    rebuild_best_json_from_trials(trials_path, best_path, "genetic_algorithm")
    display(pd.read_csv(trials_path).sort_values(["validation_macro_f1", "best_validation_loss"], ascending=[False, True]).head())
else:
    print("RUN_GENETIC_ALGORITHM is False. [RESULT TO BE INSERTED AFTER FINAL RUN]")
    print("Missing best config:", best_path)
    print("Missing trial CSV:", trials_path)


Epoch 1/50
36/36 - 12s - 345ms/step - accuracy: 0.8369 - loss: 0.3854 - val_accuracy: 0.8996 - val_loss: 0.2435
Epoch 2/50
36/36 - 0s - 5ms/step - accuracy: 0.8954 - loss: 0.2387 - val_accuracy: 0.9222 - val_loss: 0.1929
Epoch 3/50
36/36 - 0s - 5ms/step - accuracy: 0.9155 - loss: 0.1898 - val_accuracy: 0.9403 - val_loss: 0.1420
Epoch 4/50
36/36 - 0s - 5ms/step - accuracy: 0.9274 - loss: 0.1682 - val_accuracy: 0.9341 - val_loss: 0.1611
Epoch 5/50
36/36 - 0s - 5ms/step - accuracy: 0.9342 - loss: 0.1517 - val_accuracy: 0.9423 - val_loss: 0.1411
Epoch 6/50
36/36 - 0s - 5ms/step - accuracy: 0.9403 - loss: 0.1362 - val_accuracy: 0.9516 - val_loss: 0.1164
Epoch 7/50
36/36 - 0s - 5ms/step - accuracy: 0.9455 - loss: 0.1281 - val_accuracy: 0.9557 - val_loss: 0.1157
Epoch 8/50
36/36 - 0s - 5ms/step - accuracy: 0.9540 - loss: 0.1135 - val_accuracy: 0.9459 - val_loss: 0.1459
Epoch 9/50
36/36 - 0s - 5ms/step - accuracy: 0.9549 - loss: 0.1066 - val_accuracy: 0.9547 - val_loss: 0.1197
Epoch 10/50
36/3

Epoch 1/50
36/36 - 8s - 234ms/step - accuracy: 0.8490 - loss: 0.3379 - val_accuracy: 0.9274 - val_loss: 0.1689
Epoch 2/50
36/36 - 0s - 5ms/step - accuracy: 0.9178 - loss: 0.1966 - val_accuracy: 0.9300 - val_loss: 0.1552
Epoch 3/50
36/36 - 0s - 5ms/step - accuracy: 0.9416 - loss: 0.1484 - val_accuracy: 0.9372 - val_loss: 0.1556
Epoch 4/50
36/36 - 0s - 5ms/step - accuracy: 0.9499 - loss: 0.1272 - val_accuracy: 0.9501 - val_loss: 0.1119
Epoch 5/50
36/36 - 0s - 5ms/step - accuracy: 0.9525 - loss: 0.1150 - val_accuracy: 0.9629 - val_loss: 0.1057
Epoch 6/50
36/36 - 0s - 5ms/step - accuracy: 0.9638 - loss: 0.0938 - val_accuracy: 0.9629 - val_loss: 0.0931
Epoch 7/50
36/36 - 0s - 5ms/step - accuracy: 0.9656 - loss: 0.0870 - val_accuracy: 0.9624 - val_loss: 0.0968
Epoch 8/50
36/36 - 0s - 5ms/step - accuracy: 0.9704 - loss: 0.0721 - val_accuracy: 0.9392 - val_loss: 0.1806
Epoch 9/50
36/36 - 0s - 5ms/step - accuracy: 0.9723 - loss: 0.0710 - val_accuracy: 0.9624 - val_loss: 0.0988
Epoch 10/50
36/36

Epoch 1/50
36/36 - 6s - 169ms/step - accuracy: 0.8365 - loss: 0.3397 - val_accuracy: 0.9001 - val_loss: 0.2273
Epoch 2/50
36/36 - 0s - 5ms/step - accuracy: 0.8992 - loss: 0.2227 - val_accuracy: 0.9202 - val_loss: 0.1800
Epoch 3/50
36/36 - 0s - 5ms/step - accuracy: 0.9188 - loss: 0.1884 - val_accuracy: 0.9135 - val_loss: 0.2012
Epoch 4/50
36/36 - 0s - 5ms/step - accuracy: 0.9266 - loss: 0.1691 - val_accuracy: 0.9274 - val_loss: 0.1674
Epoch 5/50
36/36 - 0s - 5ms/step - accuracy: 0.9336 - loss: 0.1576 - val_accuracy: 0.9264 - val_loss: 0.1679
Epoch 6/50
36/36 - 0s - 5ms/step - accuracy: 0.9364 - loss: 0.1499 - val_accuracy: 0.9212 - val_loss: 0.1889
Epoch 7/50
36/36 - 0s - 5ms/step - accuracy: 0.9417 - loss: 0.1378 - val_accuracy: 0.9418 - val_loss: 0.1403
Epoch 8/50
36/36 - 0s - 5ms/step - accuracy: 0.9458 - loss: 0.1274 - val_accuracy: 0.9392 - val_loss: 0.1376
Epoch 9/50
36/36 - 0s - 5ms/step - accuracy: 0.9474 - loss: 0.1227 - val_accuracy: 0.9470 - val_loss: 0.1231
Epoch 10/50
36/36

## 5. Checks


In [6]:
# Purpose: Runs this notebook step and prints or saves the resulting intermediate output for review.
display(pd.DataFrame([("unique_budget", 40), ("same_domain_as_random_search", True), ("duplicate_configs_reuse_fitness", True), ("test_metrics_computed", False)], columns=["check", "value"]))


,check,value
0,unique_budget,40
1,same_domain_as_random_search,True
2,duplicate_configs_reuse_fitness,True
3,test_metrics_computed,False
